# 基于Add算子的核函数介绍

在前文中，我们系统性地介绍了Ascend C算子编程范式与核心API。其中，基本的矢量编程范式将算子的计算流程清晰地划分为三个串行流水任务：CopyIn、Compute与CopyOut。这三个任务通过TQue队列进行数据接力，并由TPipe统一管理与分配内存及同步资源，共同构成一个高效的流水线。本章将在前文基础上，完整呈现一个矢量Add算子从分析、实现到调用的全过程。内容涵盖：算子分析、核函数定义、根据编程范式实现算子类，以及从Host侧编写应用程序调用核函数并完成最终交付。通过本章学习，您将掌握Ascend C算子开发的完整流程。

本章学习大纲如下：

- 算子分析：分析算子的数学表达式、输入、输出以及计算逻辑的实现，明确需要调用的Ascend C接口。
- 核函数定义：定义Ascend C算子入口函数。
- 根据编程范式实现算子类：完成核函数的内部实现
- 核函数调用代码的开发：编写Host侧应用程序调用核函数。
- 核函数执行程序的交付件组成：了解最终生成的可交付程序文件。
---
## 1. 环境准备

正式开始学习之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用bisheng编译器，完成算子的开发及编译。

In [18]:
!mkdir -p Sources/02.04

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

---

## 2. 算子分析

在实现核函数前，首先需要明确算子的计算逻辑、数据需求及所需调用的Ascend C编程接口。下文以Add算子为例，展示这一分析过程。

- 1.明确算子的数学表达式及计算逻辑。

    Add算子的数学表达式为：

    $\vec{z}$ =$\vec{x}$ +$\vec{y}$

    计算逻辑是数据需经历“搬运-计算-搬运”三个阶段：
    - 将输入数据从Global Memory搬运至Local Memory。
    - 在Local Memory中使用Ascend C计算接口执行矢量加法计算。
    - 将计算结果从Local Memory搬运回Global Memory。

- 2.明确输入和输出。
    - 输入数量：2个，分别为 x 和 y。
    - 输出数量：1个，为 z。
    - 数据类型：输入与输出均为 float 类型。
    - 数据形状（Shape）： 输入Shape为 (8, 2048)，输出Shape与输入相同。
    - 数据布局（Format）：ND。

- 3.确定核函数名称和参数。

    根据输入输出分析，确定核函数参数，定义核函数原型：
    - 核函数名称：add_custom
    - 参数列表：
    - x, y: 输入数据在Global Memory上的起始地址。
    - z: 输出数据在Global Memory上的起始地址。

- 4.确定算子实现所需接口。

    基于实现流程，确定需要调用的关键API类别：
    - 数据搬运：使用 DataCopy 接口完成Global与Local Memory间的数据搬移。
    - 矢量计算：使用矢量 Add 接口实现 x 与 y 的逐元素相加。
    - 内存管理：使用 AllocTensor 与 FreeTensor 为LocalTensor申请和释放内存。
    - 任务同步：使用 EnQue、DeQue 接口实现流水任务间的数据通信与同步。

通过以上分析，得到Ascend C Add算子的设计规格如下：

<img src="./images/ascend_c_add_operator_design_specification.png" alt="ascend_c_add_operator_design_specification"  width="800px" >

---

## 3. 核函数开发

基于对Add算子的分析，我们开始着手实现核函数代码。Ascend C算子开发遵循一套固定的代码组织模式：**核函数（Kernel）**作为设备侧对外的执行入口，调用内部定义的**算子类（Kernel Class）**，该类则严格遵循编程范式，在**Init**和**Process**函数中完成全部计算逻辑。以下代码均在[add_custom.asc](./src/add_custom.asc)中完整实现，这里重点在于理解**多核并行**与**单核内流水并行**的协同设计思想。

### 3.1 头文件引入
进行算子开发时，首先要在add_custom.asc源文件中导入必要的头文件。这些头文件都是固定头文件，在进行其它自定义算子开发时可直接复用。  

- Host侧应用程序需要包含的头文件：#include "acl/acl.h"

- Kernel侧需要包含的头文件：#include "kernel_operator.h"

In [ ]:
%%writefile Sources/02.04/add_custom.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

### 3.2 BufferNum定义

在[02.03_AscendC编程范式介绍与API介绍](./02.03_ascendc_programming_paradigm_and_api_introduction.ipynb)中，我们知道DoubleBuffer是基于MTE指令队列与Vector指令队列的独立性和可并行性，通过将数据搬运与Vector计算并行执行以隐藏数据搬运时间并降低Vector指令的等待时间，最终提高Vector单元的利用效率，可以通过**为队列申请内存时设置内存块的个数**来实现数据并行，使用Double Buffer的简单代码示例如下：
```
pipe.InitBuffer(inQueueX, 2, 256);
```
而在实际代码中，一般会用宏进行替换，实际代码如下：

In [ ]:
%%writefile -a Sources/02.04/add_custom.asc

constexpr uint32_t BUFFER_NUM = 2; // tensor num for each queue

### 3.3 自定义结构体AddCustomTilingData

在启动核函数之前，主机侧（Host）需要明确告知设备侧（Device）如何对总数据进行划分，以实现多核并行与单核内流水并行。这一策略通过一个自定义的结构体 **AddCustomTilingData** 来传递。该结构体的意义与作用：

- **解耦与抽象**：它将数据切分的策略参数化，使核函数逻辑与具体的切分方案解耦。若要调整切分方式（如改变tileNum以优化性能），只需修改主机侧传入的参数，无需改动核函数代码。

- **信息传递**：作为核函数的参数，它将主机侧决定的全局数据视图（totalLength）和单核并行度（tileNum）准确地传递给每个核上执行的核函数实例。

- **两级数据并行计算的桥梁**：

    - totalLength 与系统启动的核数（GetBlockNum()，例如8）共同决定了**第一级多核并行的粒度** blockLength（即totalLength / GetBlockNum()）。

    - tileNum 直接决定了**第二级单核内流水并行的粒度**，指导核函数内部如何为TPipe和TQue分配内存，以及循环计算多少次。

有了这个结构体，核函数的定义就更加清晰完整。它作为核函数的参数之一，确保所有核根据统一的规划协同工作。

In [ ]:
%%writefile -a Sources/02.04/add_custom.asc

// 描述数据并行切分策略的结构体，由主机侧填充并传递给核函数
struct AddCustomTilingData
{
    uint32_t totalLength; // 待处理数据总长度（单位：元素个数），例如 8 * 2048
    uint32_t tileNum;     // 单核内将数据进一步切分为多少块（Tile），例如 8
};

### 3.4  实现核函数入口

根据[02.02_HelloWorld](./02.02_HelloWorld.ipynb)介绍的规则进行核函数的定义，在核函数中调用算子类的Init和Process函数，算子类实现在后续步骤中介绍。

In [ ]:
%%writefile Sources/02.04/add_custom_bak.asc

// __global__ 和 __aicore__ 限定符表明这是一个在AI Core上执行的设备核函数
__global__ __aicore__ void add_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z, AddCustomTilingData tiling) {
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY); // 声明为矢量计算内核
    KernelAdd op;                                   // 实例化算子类
    op.Init(x, y, z, tiling.totalLength, tiling.tileNum); // 初始化，传入切分策略
    op.Process();                                   // 执行核心计算流水线
}

**关键说明：**

- \_\_global__ \_\_aicore__：标识这是一个在AI Core上执行的核函数

- GM_ADDR：宏定义，修饰指向Global Memory的指针（#define GM_ADDR \__gm__ uint8_t*）

- AddCustomTilingData：包含数据切分参数的结构体

接下来，算子类 KernelAdd 的 Init 函数将利用 tiling 中的参数，完成具体的内存分配与地址计算。

### 3.5 根据矢量编程范式实现算子类KernelAdd

算子类KernelAdd是实现的载体，其成员变量和函数精确对应了矢量编程范式所需的任务、队列与资源。

In [ ]:
%%writefile -a Sources/02.04/add_custom.asc

class KernelAdd {
public:
    __aicore__ inline KernelAdd(){}
    // 初始化函数，完成内存初始化相关操作
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum);
    // 核心处理函数，实现算子逻辑，调用私有成员函数CopyIn、Compute、CopyOut完成矢量算子的三级流水操作
    __aicore__ inline void Process();

private:
    // 搬入函数，从Global Memory搬运数据至Local Memory，被核心Process函数调用
    __aicore__ inline void CopyIn(int32_t progress);
    // 计算函数，完成两个输入参数相加，得到最终结果，被核心Process函数调用
    __aicore__ inline void Compute(int32_t progress);
    // 搬出函数，将最终结果从Local Memory搬运到Global Memory上，被核心Process函数调用
    __aicore__ inline void CopyOut(int32_t progress);

private:
    // 核心数据成员
    AscendC::TPipe pipe;  // TPipe内存管理对象
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;  // 输入数据Queue队列管理对象，TPosition为VECIN
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;  // 输出数据Queue队列管理对象，TPosition为VECOUT
    AscendC::GlobalTensor<float> xGm;  // 管理输入输出Global Memory内存地址的对象，其中xGm, yGm为输入，zGm为输出
    AscendC::GlobalTensor<float> yGm;
    AscendC::GlobalTensor<float> zGm;
    // 数据切分参数
    uint32_t blockLength; // 每个核的计算数据长度
    uint32_t tileNum; // 每个核需要计算的数据块个数
    uint32_t tileLength; // 每个核内每个数据块的长度
};


内部函数的调用关系示意图如下：

<img src="./images/kernel_function_call_relationship_diagram.png" alt="kernel_function_call_relationship_diagram"  width="500px" >

由此可见Init函数完成了初始化，Process中完成了对流水任务“搬入、计算、搬出”的调用。

### 3.6 初始化函数Init实现多级数据切分

Init函数负责完成**数据切分与内存分配**，这是实现并行计算的关键。

我们需要理解两级并行设计：

- **第一级：多核并行**，数据在不同AI Core间切分，并行计算
- **第二级：单核内流水并行**，数据在单个核内进一步切块，支持流水线并行执行


1.**多核数据切分**：跨核的负载均衡

假设总数据量为8 * 2048个元素，使用8个核并行处理，每个核处理2048个元素。多核切分的关键即调用AscendC::GetBlockIdx()返回当前核的索引（0-7），通过blockLength * GetBlockIdx()计算偏移，确保**每个核处理不同的数据段**。这样也就实现了多核并行计算的数据切分。

以输入x为例，x + blockLength * GetBlockIdx()即为单核处理程序中x在Global Memory上的内存偏移地址，获取偏移地址后，使用GlobalTensor类的SetGlobalBuffer接口设定该核上Global Memory的起始地址以及长度，具体示意图如下。

<img src="./images/multi_core_parallel_processing_schematic.png" alt="multi_core_parallel_processing_schematic"  width="700px" >    

2.**单核内数据切块**：流水线并行的基础

对于单核上的处理数据，可以进行数据切块（Tiling），以实现流水线并行，这里仅作为参考，将数据切分成8块（并不意味着8块就是性能最优）。切分后的每个数据块再次切分成2块，即可开启**double buffer**，实现流水线之间的并行。

单核数据切分示意图：

<img src="./images/single_core_data_slicing_schematic.png" alt="single_core_data_slicing_schematic"  width="700px" >

总数据(2048) → 8个Tile → 每个Tile再分为2块（Double Buffer）

这样单核上的数据（2048个数）被切分成16块，每块tileLength（128）个数据。

还是以输入x为例，TPipe为inQueueX分配了两块大小为tileLength * sizeof(float)个字节的内存块，每个内存块能容纳tileLength（128）个float类型数据。

这种切分带来两个优势：

- 流水线并行：不同数据块在不同处理阶段同时进行

- Double Buffer：隐藏数据搬运延迟，提升效率

具体的初始化函数代码如下：

In [ ]:
%%writefile -a Sources/02.04/add_custom.asc

__aicore__ inline void KernelAdd::Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
{
    // 计算每个核处理的数据长度（多核并行）
     this->blockLength = totalLength / AscendC::GetBlockNum();     // length computed of each core
    // 记录单核内数据切块参数
     this->tileNum = tileNum;                                      // split data into 8 tiles for each core
     this->tileLength = this->blockLength / tileNum / BUFFER_NUM;  // separate to 2 parts, due to double buffer
    // 设置每个核的Global Memory起始地址（关键的多核切分逻辑）
     xGm.SetGlobalBuffer((__gm__ float *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     yGm.SetGlobalBuffer((__gm__ float *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     zGm.SetGlobalBuffer((__gm__ float *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
    // 为队列分配内存（单核内流水并行的基础）
     pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(float));
}


### 3.7 核心处理流程Process函数实现

基于矢量编程范式，将核函数的实现分为3个基本任务：CopyIn，Compute，CopyOut。Process函数中通过如下方式调用这三个函数，实现计算与数据搬运的重叠：

In [ ]:
%%writefile -a Sources/02.04/add_custom.asc

__aicore__ inline void KernelAdd::Process()
{
    // 循环次数 = tileNum × BUFFER_NUM（考虑Double Buffer）
    int32_t loopCount = this->tileNum * BUFFER_NUM;
    // 流水线并行：依次处理每个数据块
    for (int32_t i = 0; i < loopCount; i++) {
        CopyIn(i);    // 搬入第i块数据
        Compute(i);   // 计算第i块数据
        CopyOut(i);   // 搬出第i块结果
    }
}

**重要说明**：由于采用Double Buffer机制，循环次数是tileNum * BUFFER_NUM（16次），而不是tileNum（8次）。这意味着同一时刻，CopyIn、Compute、CopyOut可能处理的是不同的数据块，实现真正的流水线并行。

流水任务的具体实现如下：

**CopyIn任务**：
1. 使用DataCopy接口将GlobalTensor数据拷贝到LocalTensor。
2. 使用EnQue将LocalTensor放入VecIn的Queue中。

In [ ]:
%%writefile -a Sources/02.04/add_custom.asc

__aicore__ inline void KernelAdd::CopyIn( int32_t progress)
{
    // 为LocalTensor分配内存
    AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.AllocTensor<float>();
    // 计算当前数据块的起始位置,从Global Memory搬入数据到Local Memory
    AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
    AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
    // 将数据放入队列，通知Compute任务
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);
}


**Compute任务**：
1. 使用DeQue从VecIn中取出LocalTensor。
2. 使用Ascend C接口Add完成矢量计算。
3. 使用EnQue将计算结果LocalTensor放入到VecOut的Queue中。
4. 使用FreeTensor将释放不再使用的LocalTensor。

In [ ]:
%%writefile -a Sources/02.04/add_custom.asc

__aicore__ inline void KernelAdd::Compute(int32_t progress)
{
    // 从输入队列获取数据
    AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.DeQue<float>();
    // 为计算结果分配内存
    AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
    // 执行矢量加法计算
    AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
    // 将计算结果放入输出队列，通知CopyOut任务
    outQueueZ.EnQue<float>(zLocal);
    // 释放输入数据内存（重要：避免内存泄漏）
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
}


**CopyOut任务**：
1. 使用DeQue接口从VecOut的Queue中取出LocalTensor。
2. 使用DataCopy接口将LocalTensor拷贝到GlobalTensor上。
3. 使用FreeTensor将不再使用的LocalTensor进行回收。

In [ ]:
%%writefile -a Sources/02.04/add_custom.asc

__aicore__ inline void KernelAdd::CopyOut(int32_t progress)
{
    // 从输出队列获取计算结果
    AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
    // 计算当前数据块的输出位置,将结果从Local Memory搬出到Global Memory
    AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
    // 释放输出数据内存
    outQueueZ.FreeTensor(zLocal);
}

### 3.8 写入核函数

由于核函数会调用实现类中的函数，所以需要将3.4中暂存于add_custom_bak.asc中的核函数代码写入add_custom.asc

In [29]:
!cat Sources/02.04/add_custom_bak.asc >> Sources/02.04/add_custom.asc
!rm Sources/02.04/add_custom_bak.asc

通过以上实现，我们完成了从算子分析到核函数代码实现的完整流程：

1. 多核并行设计：通过GetBlockIdx()和blockLength实现数据在不同AI Core间的切分

2. 单核内流水设计：通过数据切块和Double Buffer实现计算与数据搬运的并行

3. 三级流水线：CopyIn、Compute、CopyOut任务通过队列协同工作

4. 统一资源管理：TPipe负责内存分配，确保资源高效利用

这种设计模式不仅适用于Add算子，也是大多数Ascend C算子的通用实现框架。掌握了这一模式，开发者可以快速构建各种高性能算子实现。

---

## 4. 核函数调用代码的开发

完成Kernel侧核函数开发后，即可编写Host侧的应用程序来调用算子并执行计算。在[2.02_HelloWorld章节](./02.02_HelloWorld.ipynb)中，我们已经学习了完整的Ascend C核函数调用流程。矢量Add算子的调用流程与HelloWorld算子的主要区别在于，它涉及真实的输入/输出数据。因此，在调用过程中需要额外完成数据准备、内存申请与拷贝，并在计算完成后将结果拷贝回Host侧进行验证。为保持代码结构清晰，我们将核函数的调用过程封装为独立的函数kernel_add，并在主函数中创建测试数据、调用该函数并验证结果。

### 4.1 kernel_add函数实现

kernel_add 函数负责在Host侧组织完整的核函数调用流程。其实现严格遵循异构计算的标准范式，主要包括以下七个步骤：
- 初始化AscendCL运行环境。
- 申请与管理运行时资源，包括Device、Context和Stream。
- 在Host侧准备输入数据，并申请Host与Device侧内存，将数据从Host拷贝至Device。
- 通过核函数调用符<<<...>>>在指定的AI Core上执行核函数。
- 将计算结果从Device侧内存拷贝回Host侧。
- 释放已申请的运行时资源。
- 去初始化AscendCL环境。

<img src="./images/add_invocation_steps.png" alt="add_invocation_steps"  width="150px" >

以下是该函数的完整实现代码，每个关键步骤均配有详细注释：

In [ ]:
%%writefile -a Sources/02.04/add_custom.asc

std::vector<float> kernel_add(std::vector<float> &x, std::vector<float> &y)
{
    // ---------- 步骤1: 参数与变量准备 ----------
    constexpr uint32_t blockDim = 8;                            // 启动的核数，与数据分片数对应
    uint32_t totalLength = x.size();                            // 总数据长度
    size_t totalByteSize = totalLength * sizeof(float);         // 数据总字节数
    int32_t deviceId = 0;                                       // 使用的设备ID
    aclrtStream stream = nullptr;                               // 计算流，用于异步任务管理
    // 核函数的切分参数 totalLength & tileNum，通知每个核如何划分数据
    AddCustomTilingData tiling = {totalLength, 8};
    // Host侧数据指针（输入数据已由调用者准备好）
    uint8_t *xHost = reinterpret_cast<uint8_t *>(x.data());
    uint8_t *yHost = reinterpret_cast<uint8_t *>(y.data());
    // 输出数据在Host侧的缓冲区
    uint8_t *zHost = nullptr;
    // Device侧数据指针
    uint8_t *xDevice = nullptr;
    uint8_t *yDevice = nullptr;
    uint8_t *zDevice = nullptr;

    // ---------- 步骤2: 初始化AscendCL运行环境 ----------
    aclInit(nullptr);

    // ---------- 步骤3: 申请运行时资源 ----------
    aclrtSetDevice(deviceId);                                   // 设置当前使用的Device
    aclrtCreateStream(&stream);                                 // 创建计算流，用于任务排队与异步执行

    // ---------- 步骤4: 内存分配 ----------
    aclrtMallocHost((void **)(&zHost), totalByteSize);          // 在Host侧为输出结果分配内存
    // 在Device侧为输入、输出数据分配内存
    aclrtMalloc((void **)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    
    // ---------- 步骤5: 数据搬运（Host -> Device） ----------
    aclrtMemcpy(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, yHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    
    // ---------- 步骤6: 核函数调用 ----------
    // 用内核调用符<<<...>>>启动核函数，指定核数、流及参数
    add_custom<<<blockDim, nullptr, stream>>>(xDevice, yDevice, zDevice, tiling);
    // 等待流中的所有任务执行完成（同步点）
    aclrtSynchronizeStream(stream);

    // ---------- 步骤7: 结果回拷（Device -> Host） ----------
    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    // 将原始字节数据转换为float类型的vector，便于后续处理
    std::vector<float> z((float *)zHost, (float *)(zHost + totalByteSize));
    
    // ---------- 步骤8: 资源释放 ----------
    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);

    // ---------- 步骤9: 环境去初始化 ---------
    aclFinalize();
    return z;   // 返回计算结果
}

### 4.2 VerifyResult函数实现

VerifyResult函数中，将Ascend C算子计算的输出结果 (output) 与主机侧通过相同逻辑计算出的预期值 (golden) 进行逐元素比对，并输出验证结果。


In [ ]:
%%writefile -a Sources/02.04/add_custom.asc

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (std::equal(golden.begin(), golden.end(), output.begin())) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
        return 1;
    }
    return 0;
}


### 4.3 Host侧验证主程序main函数

main函数负责定义输入数据长度、准备测试数据、调用算子并完成验证。


In [ ]:
%%writefile -a Sources/02.04/add_custom.asc

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 1.2f;
    constexpr float valueY = 2.3f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);

    std::vector<float> output = kernel_add(x, y);

    std::vector<float> golden(totalLength, valueX + valueY);
    return VerifyResult(output, golden);
}

### 4.4 CMake编译配置
一个完整的算子应用程序需要通过编译生成可执行文件。对于Ascend C项目，我们可以使用CMake作为构建系统，并通过特定的配置来调用Ascend C编译工具链。完整的编译配置与说明如下：

In [ ]:
%%writefile Sources/02.04/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
# find_package(ASC)是CMake中用于查找和配置Ascend C编译工具链的命令
find_package(ASC REQUIRED)
# 指定项目支持的语言包括ASC和CXX，ASC表示支持使用毕昇编译器对Ascend C编程语言进行编译
project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    add_custom.asc
)
# 通过编译选项设置NPU架构
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>
)

配置要点说明：

- 工具链依赖：核心是 find_package(ASC)，它自动配置了将 .asc 文件编译为NPU可执行代码所需的所有规则。

- 双语言支持：通过 project(… LANGUAGES ASC CXX) 声明，CMake能分别用正确的编译器处理Host侧的C++代码和Device侧的Ascend C核函数代码。

- 架构指定：--npu-arch 选项至关重要，必须与运行环境的NPU型号匹配，以确保生成的二进制指令能够正确执行。昇腾AI处理器型号和__NPU_ARCH__的对应关系如下表：

    <table style="margin: 0; margin-right: auto; border-collapse: collapse;">
    <tr>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:300px;">昇腾AI处理器型号</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:100px;">__NPU_ARCH__</th>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">Atlas A3 训练系列产品/Atlas A3 推理系列产品</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2201</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">Atlas A2 训练系列产品/Atlas A2 推理系列产品</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2201</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">Atlas 200I/500 A2 推理产品</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3002</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">Atlas 推理系列产品</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2002</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">Atlas 训练系列产品</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1001</td>
    </tr>
    </table>

此配置提供了一个最小且完整的编译框架。在实际的大型项目中，可能还需要添加头文件路径、链接库、以及其他针对不同构建类型（Debug/Release）的优化选项。



### 4.5 编译和运行

执行以下命令编译可执行文件：

In [ ]:
!cd Sources/02.04 && \
mkdir -p build
!export ASC_DIR=$ASCEND_HOME_PATH/aarch64-linux/tikcpp/ascendc_kernel_cmake/ && \
cd Sources/02.04/build/ && \
cmake .. && \
make

再执行以下代码，进行算子的实际运行：

In [ ]:
!cd Sources/02.04/build/ && ./demo

--- 
## 5. 核函数执行程序的交付件组成

通过前文的学习，我们完成了矢量Add算子从核函数开发、Host侧调用到编译运行的完整流程。一个可独立编译、运行的Ascend C算子程序，其交付件通常由以下两个核心文件构成：

1. **add_custom.asc**

    此为项目的核心源码文件，采用.asc扩展名以支持Ascend C语法。它实际上是一个混合了设备侧（Device）核函数与主机侧（Host）调用代码的源文件，具体包含：
    - 核函数实现：\_\_global__ \_\_aicore__ void add_custom(...) 函数及其内部调用的算子类 KernelAdd，这部分的代码将在AI Core上执行。

    - Host侧应用程序：封装调用逻辑的 kernel_add 函数、验证函数 VerifyResult 以及程序入口 main 函数，这部分的代码在CPU上执行。

    这种将设备代码与主机代码组织在同一个.asc文件中的方式是Ascend C开发的常见实践，便于管理一个完整的算子样例。

2. **CMakeLists.txt**

    此为项目的构建配置文件，它定义了如何将 add_custom.asc 中的混合代码正确地分别编译（为设备代码生成NPU指令，为主机代码生成CPU指令）并链接成一个完整的可执行程序（如 demo）。

add_custom.asc 和 CMakeLists.txt 共同构成了一个完整、可交付的算子工程。前者提供了从计算到调用的全部源代码逻辑，后者则提供了将其构建为可执行程序的“配方”。理解这两个文件的分工，是进行项目移植、复用和集成的基础。





---
## 课后实践

请根据本节教程内容，在下面的代码框中补充实现sub_custom.asc，使用Ascend C编程实现矢量减法算子sub_custom。
- 数据类型：输入与输出均为 float 类型。
- 数据形状（Shape）： 输入Shape为 (8, 2048)，输出Shape与输入相同。
- 数据布局（Format）：ND。

In [ ]:
%%writefile Sources/02.04/sub_custom.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

constexpr uint32_t BUFFER_NUM = 2; // tensor num for each queue

struct SubCustomTilingData
{
    uint32_t totalLength;
    uint32_t tileNum;
};

class KernelSub {
public:
    __aicore__ inline KernelSub(){}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        // 请补充……
    }
    __aicore__ inline void Process()
    {
        // 请补充……
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        // 请补充……
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        // 请补充……
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        // 请补充……
    }

private:
    // 请补充……
};

__global__ __aicore__ void sub_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z, SubCustomTilingData tiling)
{
    // 请补充……
}

std::vector<float> kernel_sub(std::vector<float> &x, std::vector<float> &y)
{
    // 请补充……
}

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (std::equal(golden.begin(), golden.end(), output.begin())) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
        return 1;
    }
    return 0;
}

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 1.2f;
    constexpr float valueY = 2.3f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);

    // 请补充……

    std::vector<float> golden(totalLength, valueX - valueY);
    return VerifyResult(output, golden);
}

完成sub_custom.asc文件开发后，执行以下命令进行编译并验证结果：

In [ ]:
%%writefile Sources/02.04/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
find_package(ASC REQUIRED)
project(kernel_samples LANGUAGES ASC CXX)

add_executable(sub_test
    sub_custom.asc
)
target_compile_options(sub_test PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>
)

In [ ]:
!cd Sources/02.04 && \
mkdir -p build
!export ASC_DIR=$ASCEND_HOME_PATH/aarch64-linux/tikcpp/ascendc_kernel_cmake/ && \
cd Sources/02.04/build/ && \
cmake .. && \
make
!cd Sources/02.04/build/ && \
./sub_test

预期执行效果如下：

<img src="./images/execution_result_04.png" alt="execution_result_04"  width="800px" >

执行以下代码获取答案。

In [ ]:
!cat ./answer/02.04_answer.txt